# 小规模CN-Celeb分数评测
本次评测的Notebook、脚本、音频、固定Fbank、日志、缓存和结果都位于evaluation文件夹。复用上一级已有的模型、运行库和55层验证代码。

4名说话人、12条完整录音、32条官方试验（8同人/24异人）。两种板卡模型使用相同的原始前处理Fbank，比较模型输出和分数。本轮不测板卡前处理、采集或速度，不校准鉴权阈值。

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, time, uuid
sys.dont_write_bytecode=True
WORK=Path.cwd().resolve()
assert WORK.name=='evaluation' and (WORK/'quality_pilot.py').is_file(), '请打开full_model_test/evaluation里的Notebook'
ROOT=WORK.parent
assert (ROOT/'runtime_core.py').is_file(), 'evaluation必须位于现有full_model_test里面'
assert platform.system()=='Linux' and platform.machine().lower() in ('armv7l','armv8l')
# 收拢以前上传到根目录的本次评测文件；只移动明确的五项，保留原件。
legacy_names=('quality_pilot.py','04a_cnceleb_score_pilot.ipynb','quality_pilot_manifest.json','QUALITY_PILOT.md','quality_pilot_data')
legacy=None
for name in legacy_names:
    source=ROOT/name
    if source.exists():
        assert not source.is_symlink() and source.resolve().parent==ROOT
        if legacy is None:
            legacy=WORK/'legacy_root_upload'/uuid.uuid4().hex
            legacy.mkdir(parents=True)
        target=legacy/name
        assert target.resolve().is_relative_to(WORK) and not target.exists()
        shutil.move(str(source),str(target))
if legacy is not None:
    print('之前散落的评测文件已归档到:',legacy)
RUN=WORK/'results'/('pilot_'+hashlib.sha256((WORK/'manifest.json').read_bytes()).hexdigest()[:16])
RUN.mkdir(parents=True,exist_ok=True)
def execute_stage(stage):
    destination=RUN/stage
    destination.mkdir(exist_ok=True)
    command=[sys.executable,'-u','-B',str(WORK/'quality_pilot.py'),'--stage',stage,'--output',str(destination)]
    if stage=='fpga':
        gate=RUN/'cpu/summary.json'
        assert gate.is_file() and json.loads(gate.read_text()).get('completed'), '请先完成CPU格'
        command+=['--cpu-directory',str(RUN/'cpu')]
    logfile=destination/'execution.log'
    print('开始:',stage,'日志:',logfile,flush=True)
    with logfile.open('w',encoding='utf-8') as writer:
        proc=subprocess.Popen(command,cwd=str(WORK),text=True,stdout=writer,stderr=subprocess.STDOUT,
            env={**os.environ,'PYTHONDONTWRITEBYTECODE':'1','PYTHONUNBUFFERED':'1'})
        deadline=time.monotonic()+7200
        try:
            with logfile.open('r',encoding='utf-8',errors='replace') as reader:
                while True:
                    chunk=reader.read()
                    if chunk:
                        print(chunk,end='',flush=True)
                    returncode=proc.poll()
                    if returncode is not None:
                        tail=reader.read()
                        if tail:
                            print(tail,end='',flush=True)
                        break
                    if time.monotonic()>deadline:
                        proc.kill()
                        proc.wait()
                        returncode=-1
                        print('\n本阶段超过7200秒，已停止。请返回日志。',flush=True)
                        if stage=='fpga':
                            print('若发生硬件超时或IP仍忙，请重启板卡后再继续。',flush=True)
                        break
                    time.sleep(2)
        finally:
            if proc.poll() is None:
                proc.kill()
                proc.wait()
    print('\n返回码:',returncode,'结果目录:',destination,flush=True)
    assert returncode==0, '本阶段未完成，请返回完整输出，不继续下一格'
print('评测结果目录:',RUN)
print('每条完成即保存；中断后重跑同一格可复用校验缓存。')


In [ ]:
execute_stage('cpu')


CPU格出现PILOT_COMPLETED后运行下一格。FPGA格会核对已有03b的55层硬件一致性通过记录，继续保留逐层整数和边界检查。

In [ ]:
execute_stage('fpga')


请返回CPU与FPGA的PILOT_METRICS，以及FPGA的PAIRED_SCORE_CHANGE。详细对照在本目录results/pilot_编号/fpga/paired_comparison.json和paired_scores.csv。小子集指标不代表完整数据集准确率，原模型漂移失败不会自动改写。